[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%3A%20NLP/08_Intro_redes_neuronales.ipynb)


# Redes neuronales — empezando por algo muy chico

En muchos cursos aparece GPT o el diagnóstico médico… aquí resolvemos algo mínimo: **predecir estatura desde el tamaño de la mano**.

Primero hacemos una **recta** (regresión lineal). Es rápido y claro.

Después mostramos que **podemos describir ese mismo trabajo como una red pequeña** y entrenarla con **TensorFlow**. Así tus alumnos ven el puente antes de redes más locas donde ya no obligamos una relación lineal.


In [ ]:
# %pip install numpy matplotlib scikit-learn tensorflow -q

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})


## Paso 1 — Regresión lineal (una recta)

La idea: \(\hat{y} = \beta_1 + \beta_2\,x\) con \(x\) = tamaño de la mano y \(\hat{y}\) = estatura predicha. **\(\beta_1\)** es el intercepto y **\(\beta_2\)** la pendiente. OLS busca la recta que mejor cuadra con los puntos (en el sentido de minimizar errores al cuadrado).

Siguiente celda: ajustar con **scikit-learn** sobre los **siete puntos**; después predices un valor nuevo y ves el gráfico.


In [ ]:
from sklearn.linear_model import LinearRegression

# --- Datos ejemplo (diagrama típico mano ↔ estatura) ---
x_mano = np.array([13.0, 16.0, 24.0, 43.0, 51.0, 84.0, 90.0])
y_estatura = np.array([23.0, 5.0, 33.0, 32.0, 53.0, 65.0, 85.0])

# sklearn quiere cada fila con sus propias características → matriz `(n_filas, 1)`.
ols_1var = LinearRegression().fit(x_mano.reshape(-1, 1), y_estatura)

# Coeficientes leíbles para el alumno cuando discutes la forma β₁ + β₂·mano.
beta1 = float(ols_1var.intercept_)
beta2 = float(ols_1var.coef_[0])

print('OLS con siete puntos — coeficientes estimados:')
print(f'  β₁ (intercepto):  {beta1:.4f}')
print(f'  β₂ (pendiente):   {beta2:.4f}')


In [ ]:
# Puedes mover `mano_nueva` cualquier tamaño físico nuevo que quieran simular tras entrenamiento.
mano_nueva = 55.0

row = np.array([[mano_nueva]])   # igual que arriba, la API exige formato fila‑columnas
estatura_pred = float(ols_1var.predict(row)[0])

print(f'Si el tamaño de la mano es {mano_nueva:g}:')
print(f'  → estatura predicha ≈ {estatura_pred:.2f}')
print(
    '(comprobación manual: β₁ + β₂×mano = '
    f'{beta1:.4f} + {beta2:.4f} × {mano_nueva:g} ≈ '
    f'{beta1 + beta2 * mano_nueva:.2f})'
)

In [ ]:
# Curva interpolada muy suave sólo muestra mejor la línea encontrada contra los dispersos violetas.
fine = np.linspace(float(x_mano.min()), float(x_mano.max()), 200)
recta = beta1 + beta2 * fine

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(x_mano, y_estatura, alpha=0.88, color='#7C3AED', edgecolors='white',
           linewidths=0.5, label='Las 7 observaciones')
ax.plot(fine, recta, color='#DC2626', linewidth=2.5,
        label='Recta ajustada: β₁ + β₂ · tam_mano')

ax.set_xlabel('Tamaño de la mano')
ax.set_ylabel('Estatura')
ax.set_title('Mínimos cuadrados sobre siete puntos')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## Paso 2 — Misma historia, vista como red neuronal

En libros aparece que muchos procedimientos se pueden dibujar como **red**. La idea práctica aquí es: **mis datos son los mismos** (filas de mano ↔ estatura), pero en vez de la fórmula cerrada del OLS vas a **entrenar**: el programa ajusta **pesos** repitiendo.

Cuando ya entrenaste, llega un alumno nuevo con su medida de mano y preguntas: **¿qué estatura predigo?** — igual que con la recta, pero después de esta rutina llamada gradiente/epocas que verán correr abajo.

Más adelante veremos redes donde **ya no obligamos una recta**. Hoy todas las activaciones son **lineales** a propósito: la familia del modelo sigue siendo cercana al caso lineal para que pegue con el paso 1. **Entrada**: un número (tamaño de mano).

En código: **entrada** = un número por fila (la mano) → **capa oculta** `Dense` con activación **lineal** → **salida** `Dense(1)` también **lineal**. Así ves la “cascarita” entrada–medio–salida como en los diagramas generales del curso.

In [ ]:
# TensorFlow y Keras: aquí sólo cargamos el ecosistema. Las dos celdas siguientes
# prepararán datos y crearán capas del modelo paso por paso.
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# Cada FILA es un ejemplo: una columna = la mano (`float32` es lo que suele esperar la red).
# `y_nn` lleva las estaturas observadas pareadas por índice con `x_mano`.
X_nn = np.asarray(x_mano, dtype=np.float32).reshape(-1, 1)
y_nn = np.asarray(y_estatura, dtype=np.float32)

tf.keras.utils.set_random_seed(42)

In [ ]:
# `Sequential`: se leen capas como un tubo izquierda → derecha.
# Una entrada escalar amplificada primero a 8 salidas LINEALES,
# después se combina todo en una sola predicción (también lineal): sigue pareciendo a una recta.
modelo_mano = keras.Sequential(
    [
        layers.Input(shape=(1,), name='entrada_mano'),
        layers.Dense(8, activation='linear', name='capa_oculta'),
        layers.Dense(1, activation='linear', name='salida_estatura'),
    ],
    name='ejemplo_recta_lineal',
)

In [ ]:
# `compile`: eliges cómo mejorar pesos (`optimizer`) y qué pérdida bajar durante el entrenamiento (`mse`).
modelo_mano.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.05),
    loss='mse',
)

`fit` = enseñar el modelo repetidas veces con los mismos pares *(mano, estatura)* y bajar la pérdida `mse` (error cuadrático medio). Ahí aparece la idea moderna del entrenamiento: medir mal desempeño, calcular gradientes, actualizar.

In [ ]:
# `epochs`: cuántas veces recorrerá el conjunto completo antes de frenar esta corrida rápida.
# `verbose=0` sólo oculta la barrita ruidosa; la historia queda igual en la variable Python.
historia = modelo_mano.fit(X_nn, y_nn, epochs=800, verbose=0)

print('Entrenamiento listo.')
print(f"Última pérdida (mse): {float(historia.history['loss'][-1]):.6f}")

### ¿Qué es ese número de `mse`?

En entrenamiento, TensorFlow está minimizando el **error cuadrático medio** sobre las **filas que le pasaste** (aquí, 7 puntos). Para cada fila \(i\):

$$
e_i^2 = (y_i - \hat{y}_i)^2
$$

donde \(e_i\) es el residual (error de predicción) de esa fila.

La pérdida que reporta `mse` es la media:

$$
\mathrm{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

Aquí \(n=7\).

En forma compacta:

$$
h = xW^{(1)} + b^{(1)}, \qquad \hat{y} = hW^{(2)} + b^{(2)}.
$$

Como todas las capas aquí son lineales y \(x\) es escalar, toda la red se reduce a una recta efectiva:

$$
\hat{y} = c_0 + c_1 x.
$$

Por eso, aunque internamente haya muchos parámetros, el comportamiento final frente a \(x\) puede compararse con la pendiente de OLS.

In [ ]:
# Extraemos pesos internos de las capas Dense para poder colapsar la red lineal a una recta.
densas_tf = [capa for capa in modelo_mano.layers if isinstance(capa, keras.layers.Dense)]
capa_oculta, capa_salida = densas_tf[0], densas_tf[1]
W1, b1 = capa_oculta.get_weights()
W2, b2 = capa_salida.get_weights()

In [ ]:
# Composición lineal de la red -> pendiente efectiva única.
pendiente_ef = float(np.dot(W1.ravel(), W2.ravel()))

print('Recta efectiva de la red lineal (resumen):')
print(f'  c1 (pendiente efectiva de la red): {pendiente_ef:.6f}')

print('\nComparación con regresión lineal (OLS):')
print(f'  β₂ OLS (pendiente):                {beta2:.6f}')

Misma persona nueva (`mano_nueva`): comparamos predicción por **recta (OLS)** y por **red**. Con capas lineales, ambas deberían dar casi el mismo valor para el mismo dato.

In [ ]:
# Inferencia: mismo formato que en sklearn — un escalar se empaqueta en matriz (1, 1) tipo float32.
mano_tf = np.array([[mano_nueva]], dtype=np.float32)
estatura_tf = float(modelo_mano.predict(mano_tf, verbose=0)[0, 0])

print(f'Mano nueva = {mano_nueva:g}')
print(f'  Estatura (OLS / recta):      {estatura_pred:.4f}')
print(f'  Estatura (red TF lineal):    {estatura_tf:.4f}')

**Cierre.** La regresión lineal fue “en un paso” (fórmula de mínimos cuadrados). La red con TensorFlow fue “en muchos pasos” pero apuntaba al mismo tipo de respuesta porque la arquitectura sigue siendo lineal. Cuando cambien las activaciones y el tamaño de la red, podrán **atacar el mismo tipo de problema sin forzar una recta** — con costos y cuidados que verán después.